In [ ]:
# Load the TSV file and convert it to CSV format
import csv
import os
from pathlib import Path

# Get project root and set paths
project_root = Path.cwd().parent.parent.parent
input_file = project_root / "data" / "raw" / "external" / "cities1000.txt"
output_file = project_root / "data" / "raw" / "external" / "output.csv"

print(f"📂 Input file: {input_file}")
print(f"📂 Output file: {output_file}")

with open(input_file, 'r', newline='', encoding='utf-8') as tsv_file:
    reader = csv.reader(tsv_file, delimiter='\t')

    with open(output_file, 'w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)

        for row in reader:
            writer.writerow(row)

print("✅ Converted TSV to CSV")

In [ ]:
# Read the CSV file into a DataFrame
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
output_file = project_root / "data" / "raw" / "external" / "output.csv"

df = pd.read_csv(output_file)
print(f"✅ Loaded CSV with shape: {df.shape}")

C:\Users\RUSHIKESH\AppData\Local\Temp\ipykernel_9736\2140867621.py:3: DtypeWarning: Columns (0: admin4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('output.csv')


In [48]:

# Define column names based on GeoNames documentation 
cols = [
    "geonameid","name","asciiname","alternatenames",
    "latitude","longitude","feature_class","feature_code",
    "country_code","cc2","admin1","admin2","admin3","admin4",
    "population","elevation","dem","timezone","modification_date"
]

df.columns = cols

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 167438 entries, 0 to 167437
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   geonameid          167438 non-null  int64  
 1   name               167437 non-null  str    
 2   asciiname          167437 non-null  str    
 3   alternatenames     136147 non-null  str    
 4   latitude           167438 non-null  float64
 5   longitude          167438 non-null  float64
 6   feature_class      167438 non-null  str    
 7   feature_code       167438 non-null  str    
 8   country_code       167358 non-null  str    
 9   cc2                96 non-null      str    
 10  admin1             167352 non-null  str    
 11  admin2             147374 non-null  str    
 12  admin3             84008 non-null   str    
 13  admin4             21410 non-null   object 
 14  population         167438 non-null  int64  
 15  elevation          28674 non-null   float64
 16  dem          

In [49]:
# Convert numeric columns
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)
df["population"] = df["population"].fillna(0).astype(int)

In [50]:
# create a alias dictionary
alias_map = {}

for _, row in df.iterrows():
    main_name = str(row["name"]).lower().strip()

    if pd.notna(row["alternatenames"]):
        alternates = row["alternatenames"].split(",")

        for alt in alternates:
            alt = alt.lower().strip()

            # Skip bad/short entries
            if len(alt) < 3:
                continue

            alias_map[alt] = main_name
            alias_map[main_name] = main_name  # Map main name to itself

print("Alias entries:", len(alias_map))


Alias entries: 776115


In [ ]:
# Update docstring and comment to reflect cleanup
# drop useless columns, normalize names, remove duplicates
df_clean = df[[
    "name", "latitude", "longitude",
    "population", "country_code"
]].copy()

# Normalize city names to lowercase
df_clean["name"] = df_clean["name"].str.lower().str.strip()

# Remove duplicate city entries
df_clean = df_clean.drop_duplicates(
    subset=["name", "latitude", "longitude"]
)

print(f"✅ Cleaned dataset shape: {df_clean.shape}")

In [ ]:
# Save cleaned dataset and alias map
import pickle
import os
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
output_dir = project_root / "data" / "processed" / "external"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

cities_file = output_dir / "cities.csv"
alias_file = output_dir / "alias_map.pkl"

df_clean.to_csv(cities_file, index=False)

# Save alias map
with open(alias_file, "wb") as f:
    pickle.dump(alias_map, f)

print(f"✅ Saved: {cities_file}")
print(f"✅ Saved: {alias_file}")

Saved: cities_clean.csv and alias_map.pkl


In [ ]:
# Demo Usage - Load cleaned data and test alias map
import pandas as pd
import pickle
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
output_dir = project_root / "data" / "processed" / "external"

cities_file = output_dir / "cities.csv"
alias_file = output_dir / "alias_map.pkl"

df = pd.read_csv(cities_file)

with open(alias_file, "rb") as f:
    alias_map = pickle.load(f)

def normalize_city(name):
    """Normalize city name using alias map."""
    name = name.lower().strip()
    return alias_map.get(name, "")

print(f"✅ Loaded cities: {len(df)} records")
print(f"✅ Loaded alias map: {len(alias_map)} entries")

In [54]:
print(normalize_city("NYC"))  # Should return "new york city"
print(normalize_city("Los Angeles"))  # Should return "los angeles"
print(normalize_city("bhir"))  # Should return "beed"

new york city
los angeles
beed
